# Extract data for a specific donor

In [ ]:
library(SeuratObject)
library(Seurat)
library(SeuratDisk)
#library(presto)
library(Matrix)
library(reticulate)
library(dplyr)

In [ ]:
st <-readRDS('data/spatial/20230911_tonsil_atlas_rna_seurat_obj.rds')
st

In [ ]:
st <- DietSeurat(st, counts = TRUE, data = FALSE, scale.data = FALSE)
st

In [ ]:
st[["RNA"]]$scale.data <- NULL
st

In [ ]:
table(st@meta.data$donor_id)

In [ ]:
donor = c('BCLL-8-T','BCLL-9-T','BCLL-10-T','BCLL-11-T','BCLL-12-T','BCLL-13-T')

In [ ]:
st <- subset(st, subset = donor_id %in% donor)

In [ ]:
table(st@meta.data$donor_id)

In [ ]:
st

In [ ]:
saveRDS(st, file = "data/spatial/scRNA.rds")

# Process scRNA

In [ ]:
st <-readRDS('data/spatial/scRNA.rds')
st

In [ ]:
colnames(st@meta.data)

In [ ]:
unique(st@meta.data$annotation_level_1)

In [ ]:
mk <- FindAllMarkers(st, group.by = 'annotation_level_1')

## Spatial 

In [ ]:
st <-readRDS('data/spatial/20220527_tonsil_atlas_spatial_seurat_obj.rds')
st

In [ ]:
counts <- st[["Spatial"]]@counts
meta <- st@meta.data[, c("barcode", "donor_id")]
images = st@images
st <- CreateSeuratObject(counts =counts , meta.data = meta)
st@images <- images
st

In [ ]:
saveRDS(st, file = "data/spatial/spatial.rds")

In [ ]:
counts <- GetAssayData(st, slot = "counts", assay = "Spatial")
writeMM(counts, "data/spatial/counts.mtx")
write.csv(rownames(counts), "data/spatial/genes.csv", row.names = FALSE)
write.csv(colnames(counts), "data/spatial/barcodes.csv", row.names = FALSE)
meta <- st@meta.data[, c("barcode", "donor_id")]
write.csv(meta, "data/spatial/meta.csv", row.names = FALSE)

In [ ]:
colnames(st@meta.data)

In [ ]:
SaveH5Seurat(st, filename = "data/spatial/spatial.h5Seurat",overwrite = TRUE)
Convert("data/spatial/spatial.h5Seurat", dest = "h5ad", overwrite = TRUE)

In [ ]:
colnames(st@meta.data)

In [ ]:
table(st@meta.data$donor_id)

In [ ]:
mat <- GetAssayData(st, assay = "Spatial", layer = "counts")
dim(mat)

In [ ]:
colnames(mat)[1:5]

In [ ]:
head(rownames(mat))

In [ ]:
meta <- st@meta.data
head(meta)
dim(meta)

In [ ]:
st@images

In [ ]:
slice <- st@images$p7hv1g_tjgmyj
coords <- slice@coordinates
head(coords)

In [ ]:
head(st@images[['tarwe1_xott6q']]@coordinates[, c("row", "col")])

In [ ]:
head(rownames(st@meta.data)) 

In [ ]:
rownames(st@meta.data) <- Cells(st)

In [ ]:
SpatialDimPlot(
  object = st,
  images = 'tarwe1_xott6q',
  group.by = 'annotation_20220215',
  pt.size.factor = 100,
)

In [ ]:
coords_list <- list()
for(name in names(st@images)) {
    slice <- st@images[[name]]    
    coords <- slice@coordinates            
    coords_list[[name]] <- coords[, c("row", "col")]         
}
all_coords <- bind_rows(coords_list)
head(all_coords)

In [ ]:
write.csv(all_coords, "data/spatial/coordinates.csv")

In [ ]:
write.csv(st@meta.data$annotation_level_1,
          file = "data/spatial/annotation_level_1.csv",
          row.names = TRUE)